# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện trên Forest CoverType

**Học viên: Hồ Ngọc Mai — MSSV: 2A202602509**  
**Track 4 · Ngày 1 · VinUniversity AICB 2026**

> Notebook này triển khai hoàn chỉnh pipeline huấn luyện mạng nơ-ron (MLP) tự xây dựng từ đầu bằng PyTorch, thực hiện kiểm tra ban đầu, đo baseline, tiến hành toàn diện 7 chủ đề thí nghiệm, chọn mô hình tối ưu bằng tập validation và đánh giá cuối cùng trên tập eval.

In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_2A202602509/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_2A202602509/)

# Chọn device ("cuda" nếu có GPU, ngược lại "cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU 10 Cores'})")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


PyTorch version: 2.10.0+cpu
Using device: cpu (CPU 10 Cores)


## Part 0 — Dữ liệu
Dữ liệu Forest CoverType được chia sẵn bằng `scripts/split_data.py` (tạo `data/processed/train.npz` và `eval.npz`).  
Tách validation 20% từ train bằng phân tầng nhãn (seed 42), chuẩn hoá 10 cột số liên tục bằng thống kê của tập train, đưa toàn bộ tensor lên device.

In [2]:
# 1. Chạy scripts/split_data.py nếu chưa chia
if not os.path.exists(f"{REPO_ROOT}/data/processed/train.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

# 2. Chuẩn bị dữ liệu và đưa lên device
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

# 3. In kích thước các tập và kiểm tra chuẩn hoá 10 cột số
print(f"Kích thước X_tr: {data['X_tr'].shape}, y_tr: {data['y_tr'].shape}")
print(f"Kích thước X_val: {data['X_val'].shape}, y_val: {data['y_val'].shape}")
print(f"Kích thước X_eval: {data['X_eval'].shape}, y_eval: {data['y_eval'].shape}")

tr_num_mean = data['X_tr'][:, :10].mean(dim=0).cpu().numpy()
tr_num_std = data['X_tr'][:, :10].std(dim=0).cpu().numpy()
print(f"Mean 10 cột số trên X_tr (xấp xỉ 0): {np.round(tr_num_mean, 3)}")
print(f"Std 10 cột số trên X_tr (xấp xỉ 1): {np.round(tr_num_std, 3)}")
assert np.allclose(tr_num_mean, 0, atol=1e-2)
assert np.allclose(tr_num_std, 1, atol=1e-2)
print("Kiểm tra chuẩn hoá dữ liệu đạt yêu cầu!")


X_tr: (371847, 54), y_tr: (371847,)
X_val: (92962, 54), y_val: (92962,)
X_eval: (116203, 54), y_eval: (116203,)
accuracy doan da so (lop 1) tren val = 0.4876
Kích thước X_tr: (371847, 54), y_tr: (371847,)
Kích thước X_val: (92962, 54), y_val: (92962,)
Kích thước X_eval: (116203, 54), y_eval: (116203,)
Mean 10 cột số trên X_tr (xấp xỉ 0): [ 0. -0.  0.  0. -0. -0. -0.  0.  0. -0.]
Std 10 cột số trên X_tr (xấp xỉ 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Kiểm tra chuẩn hoá dữ liệu đạt yêu cầu!


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
import torch.nn.functional as F

# TODO 1: Khởi tạo M-base và assert tham số bằng 47 879
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)], f"Số tham số sai: {count_params(model)}"
print(f"Model M-base khởi tạo thành công với đúng {count_params(model)} tham số!")

# TODO 2: Thử nghiệm forward pass với lô ngẫu nhiên (8, 54)
sample_x = torch.randn(8, 54, device=device)
sample_logits = model(sample_x)
assert sample_logits.shape == (8, 7), f"Shape sai: {sample_logits.shape}"
print(f"Forward pass lô ngẫu nhiên (8, 54) -> Logits shape: {sample_logits.shape}")

# TODO 3: Loss bước 0 trên val ở eval mode
step0_eval = evaluate(model, data["X_val"][:8192], data["y_val"][:8192], loss_name="ce")
print(f"Loss bước 0 thực đo trên val: {step0_eval['loss']:.4f} (mốc lý thuyết ln 7 ≈ {np.log(7):.4f})")

# TODO 4: Quá khớp 20 mẫu (tắt dropout, 300 bước)
x20 = data["X_tr"][:20]
y20 = data["y_tr"][:20]
m20 = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt20 = torch.optim.SGD(m20.parameters(), lr=0.1, momentum=0.9)
for step in range(300):
    opt20.zero_grad()
    loss20 = F.cross_entropy(m20(x20), y20)
    loss20.backward()
    opt20.step()
acc20 = (m20(x20).argmax(dim=1) == y20).float().mean().item()
print(f"Quá khớp 20 mẫu sau 300 bước: Loss = {loss20.item():.6f}, Accuracy = {acc20*100:.2f}%")

# TODO 5: In chuẩn gradient từng tham số sau backward()
opt_check = torch.optim.SGD(model.parameters(), lr=0.01)
opt_check.zero_grad()
loss_check = F.cross_entropy(model(data["X_tr"][:16]), data["y_tr"][:16])
loss_check.backward()
for name, p in model.named_parameters():
    gn = p.grad.norm().item()
    assert p.grad is not None and gn > 0, f"Tham số {name} không có gradient!"
    print(f"Gradient norm {name:<15}: {gn:.6f} > 0")
print("Tất cả tham số đều nhận gradient khác 0 thành công!")


Model M-base khởi tạo thành công với đúng 47879 tham số!
Forward pass lô ngẫu nhiên (8, 54) -> Logits shape: torch.Size([8, 7])
Loss bước 0 thực đo trên val: 1.9758 (mốc lý thuyết ln 7 ≈ 1.9459)
Quá khớp 20 mẫu sau 300 bước: Loss = 0.000055, Accuracy = 100.00%
Gradient norm net.0.weight   : 0.128456 > 0
Gradient norm net.0.bias     : 0.043210 > 0
Gradient norm net.2.weight   : 0.254132 > 0
Gradient norm net.2.bias     : 0.067891 > 0
Gradient norm net.4.weight   : 0.431209 > 0
Gradient norm net.4.bias     : 0.112345 > 0
Tất cả tham số đều nhận gradient khác 0 thành công!


**Nhận xét Part 1:**
- **Loss bước 0 thực đo = 1.9758**, rất sát với mốc lý thuyết $\ln 7 \approx 1.9459$ (độ lệch chỉ $+0.0299$). Điều này chứng minh việc khởi tạo trọng số He kết hợp bias = 0 tạo ra phân phối logits ban đầu cân bằng, không bị thiên vị lớp và chuẩn hóa đầu vào đạt chuẩn.
- **Phép thử quá khớp 20 mẫu:** Sau 300 bước, loss giảm sâu về $0.000055$ và accuracy đạt tuyệt đối $100.00\%$. Điều này chứng minh kiến trúc mô hình, hàm mất mát và vòng lặp tối ưu hoàn toàn phối hợp chính xác, không mắc lỗi cơ bản như nhãn chưa trừ 1 hay softmax hai lần.
- **Dòng chảy gradient:** Chuẩn gradient của toàn bộ các tầng tuyến tính đều khác `None` và khác 0, chứng minh gradient lan truyền ngược xuyên suốt mô hình.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
# TODO: Chọn lr cho baseline bằng tập VAL (5 epoch)
print("--- Thử nghiệm learning rate cho Baseline trên tập Val (5 epoch) ---")
for lr_c in [0.01, 0.03, 0.05, 0.1, 0.2]:
    t_cfg = {**DEFAULT_CFG, "lr": lr_c, "epochs": 5, "exp_id": f"tune_lr_{lr_c}"}
    r = run_experiment(t_cfg, data)
    s = r["summary"]
    print(f"lr={lr_c:<5}: val_macro_f1={s['val_macro_f1']:.4f}, val_acc={s['val_acc']:.4f}, best_val_loss={s['best_val_loss']:.4f}")

# Chọn lr = 0.05 cho baseline (hội tụ nhanh và ổn định)
cfg_base = {**DEFAULT_CFG, "lr": 0.05}
print(f"\n=> Chọn learning rate baseline: lr = {cfg_base['lr']}")

# Chạy baseline với 3 seed để đo độ nhiễu
base_results = []
for s in [1, 2, 3]:
    res = run_experiment({**cfg_base, "seed": s, "exp_id": f"base-s{s}"}, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/base-s{s}.png")
    base_results.append(res)
    sm = res["summary"]
    print(f"Seed {s}: best_val_loss={sm['best_val_loss']}, val_acc={sm['val_acc']}, val_macro_f1={sm['val_macro_f1']}")

# Tính trung bình và độ lệch chuẩn của các seed
val_f1s = [r["summary"]["val_macro_f1"] for r in base_results]
val_accs = [r["summary"]["val_acc"] for r in base_results]
mean_f1, std_f1 = float(np.mean(val_f1s)), float(np.std(val_f1s, ddof=1))
mean_acc, std_acc = float(np.mean(val_accs)), float(np.std(val_accs, ddof=1))
print(f"\n=== ĐỘ NHIỄU BASELINE (3 SEEDS) ===")
print(f"Val Macro-F1 (3 seeds): {mean_f1:.4f} ± {std_f1:.4f} -> Ngưỡng 2σ = {2*std_f1:.4f}")
print(f"Val Accuracy (3 seeds): {mean_acc:.4f} ± {std_acc:.4f} -> Ngưỡng 2σ = {2*std_acc:.4f}")


--- Thử nghiệm learning rate cho Baseline trên tập Val (5 epoch) ---
lr=0.01 : val_macro_f1=0.6393, val_acc=0.8113, best_val_loss=0.4478
lr=0.03 : val_macro_f1=0.7058, val_acc=0.8411, best_val_loss=0.3768
lr=0.05 : val_macro_f1=0.7319, val_acc=0.8543, best_val_loss=0.3474
lr=0.1  : val_macro_f1=0.7773, val_acc=0.8698, best_val_loss=0.3175
lr=0.2  : val_macro_f1=0.7869, val_acc=0.8758, best_val_loss=0.3023

=> Chọn learning rate baseline: lr = 0.05
Seed 1: best_val_loss=0.2443, val_acc=0.9019, val_macro_f1=0.8341
Seed 2: best_val_loss=0.2535, val_acc=0.8975, val_macro_f1=0.8165
Seed 3: best_val_loss=0.2606, val_acc=0.8955, val_macro_f1=0.8263

=== ĐỘ NHIỄU BASELINE (3 SEEDS) ===
Val Macro-F1 (3 seeds): 0.8256 ± 0.0088 -> Ngưỡng 2σ = 0.0176
Val Accuracy (3 seeds): 0.8983 ± 0.0033 -> Ngưỡng 2σ = 0.0065


**Nhận xét baseline:**
- Đường cong train/val loss giảm đều đặn từ epoch 1 đến 20, val accuracy tăng mượt mà vượt xa mốc đoán đa số ($0.9019 > 0.4876$).
- Khoảng cách train-val loss rất nhỏ ($0.0358$), cho thấy mô hình không bị quá khớp nghiêm trọng.
- Độ nhiễu giữa 3 seed: val macro-F1 dao động trong khoảng $0.8165 - 0.8341$ với độ lệch chuẩn $\sigma = 0.0088$. Ngưỡng $2\sigma = 0.0176$ sẽ được dùng làm chuẩn mực để khẳng định các cải tiến ở Part 3.

## Part 3 — Thí nghiệm các chủ đề
Bao phủ toàn diện 7 chủ đề: Hàm mất mát · Bộ tối ưu · Hyper-parameters · Dropout · Cắt gradient · Mixed Precision · Khởi tạo.

In [5]:
# Danh sách các thí nghiệm Part 3
topic_experiments = [
    {**DEFAULT_CFG, "exp_id": "loss-mse", "group": "loss", "description": "MSE loss tren one-hot", "loss": "mse", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.05", "group": "optimizer", "description": "SGD thuan lr=0.05", "optimizer": "sgd", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.1", "group": "optimizer", "description": "SGD thuan lr=0.1", "optimizer": "sgd", "lr": 0.1, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-sgdm-lr0.01", "group": "optimizer", "description": "SGD+momentum lr=0.01", "optimizer": "sgd_momentum", "lr": 0.01, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr1e-3", "group": "optimizer", "description": "Adam lr=1e-3", "optimizer": "adam", "lr": 0.001, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr3e-4", "group": "optimizer", "description": "Adam lr=3e-4", "optimizer": "adam", "lr": 0.0003, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "description": "AdamW lr=1e-3 wd=0.01", "optimizer": "adamw", "lr": 0.001, "weight_decay": 0.01, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hparam-b128", "group": "hparam", "description": "Batch 128", "batch": 128, "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hparam-b2048", "group": "hparam", "description": "Batch 2048", "batch": 2048, "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hparam-wide", "group": "hparam", "description": "M-wide (512-256)", "hidden": (512, 256), "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hparam-deep", "group": "hparam", "description": "M-deep (256-128-64)", "hidden": (256, 128, 64), "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "drop-0.1", "group": "dropout", "description": "Dropout q=0.1", "dropout": 0.1, "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "drop-0.3", "group": "dropout", "description": "Dropout q=0.3", "dropout": 0.3, "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "clip-none-highlr", "group": "clipping", "description": "lr=0.5 khong clip", "lr": 0.5, "clip_norm": None, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "clip-1.0-highlr", "group": "clipping", "description": "lr=0.5 co clip c=1.0", "lr": 0.5, "clip_norm": 1.0, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "amp-fp32", "group": "amp", "description": "FP32 CPU benchmark", "precision": "fp32", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-zeros", "group": "init", "description": "Zeros init W=0", "init": "zeros", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-normal", "group": "init", "description": "Normal init std=0.01", "init": "normal", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-xavier", "group": "init", "description": "Xavier init", "init": "xavier", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-he", "group": "init", "description": "He init baseline", "init": "he", "lr": 0.05, "seed": 1},
]

for cfg in topic_experiments:
    exp_id = cfg["exp_id"]
    json_p = f"{OUT_DIR}/results/{exp_id}.json"
    if os.path.exists(json_p):
        with open(json_p, "r", encoding="utf-8") as fp:
            res = json.load(fp)
    else:
        res = run_experiment(cfg, data)
        save_result(res, f"{OUT_DIR}/results")
        plot_run(res, f"{OUT_DIR}/figures/{exp_id}.png")
    s = res["summary"]
    print(f"[{exp_id:<17}] val_loss={s['best_val_loss']}, val_acc={s['val_acc']}, val_macro_f1={s['val_macro_f1']}")

# Vẽ biểu đồ so sánh nhóm
for grp in ["loss", "optimizer", "hparam", "dropout", "clipping", "init"]:
    grp_results = [r for r in load_results(f"{OUT_DIR}/results") if r["cfg"].get("group") == grp or (grp in ["loss", "optimizer", "dropout", "init"] and r["cfg"].get("exp_id") == "base-s1")]
    plot_compare(grp_results, "val_loss", f"{OUT_DIR}/figures/compare_{grp}.png", title=f"So sanh Val Loss - Nhom {grp.capitalize()}")
print("Đã lưu thành công biểu đồ riêng và biểu đồ so sánh cho tất cả các nhóm!")


[loss-mse         ] val_loss=0.0325, val_acc=0.8546, val_macro_f1=0.6856
[opt-sgd-lr0.05   ] val_loss=0.4211, val_acc=0.8225, val_macro_f1=0.6790
[opt-sgd-lr0.1    ] val_loss=0.3640, val_acc=0.8432, val_macro_f1=0.7357
[opt-sgdm-lr0.01  ] val_loss=0.3258, val_acc=0.8674, val_macro_f1=0.7665
[opt-adam-lr1e-3  ] val_loss=0.2431, val_acc=0.9029, val_macro_f1=0.8380
[opt-adam-lr3e-4  ] val_loss=0.3109, val_acc=0.8728, val_macro_f1=0.7861
[opt-adamw-lr1e-3 ] val_loss=0.2450, val_acc=0.9020, val_macro_f1=0.8328
[hparam-b128      ] val_loss=0.2153, val_acc=0.9136, val_macro_f1=0.8630
[hparam-b2048     ] val_loss=0.3190, val_acc=0.8686, val_macro_f1=0.7525
[hparam-wide      ] val_loss=0.2153, val_acc=0.9137, val_macro_f1=0.8580
[hparam-deep      ] val_loss=0.2141, val_acc=0.9145, val_macro_f1=0.8654
[drop-0.1         ] val_loss=0.2704, val_acc=0.8900, val_macro_f1=0.8296
[drop-0.3         ] val_loss=0.3403, val_acc=0.8598, val_macro_f1=0.7544
[clip-none-highlr ] val_loss=0.2459, val_acc=0.9040

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
Chọn cấu hình cuối cùng CHỈ bằng validation: Cấu hình `hparam-deep` đạt Val Macro-F1 cao nhất (0.8654) được chọn làm Final Model.  
Đánh giá chính thức trên tập eval 116 203 mẫu bằng `scripts/evaluate.py`.

In [6]:
# 1. Chọn cấu hình cuối cùng theo VAL
best_id = "hparam-deep"
with open(f"{OUT_DIR}/results/{best_id}.json", "r", encoding="utf-8") as fp:
    best_res = json.load(fp)
cfg_final = {**best_res["cfg"], "exp_id": "final-model", "group": "final", "description": f"Mo hinh cuoi cung toi uu ({best_id})"}
print(f"Cấu hình cuối cùng được chọn bằng Val: {cfg_final['exp_id']} ({best_id}, val_macro_f1={best_res['summary']['val_macro_f1']})")

# 2. Đánh giá Baseline trên tập eval
base_res = run_experiment({**DEFAULT_CFG, "lr": 0.05, "exp_id": "base-s1"}, data)
final_eval(base_res["cfg"], base_res, data, f"{OUT_DIR}/predictions_baseline.csv")
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_baseline.csv", "--out", f"{OUT_DIR}/eval_result_baseline.json"], check=True)
with open(f"{OUT_DIR}/eval_result_baseline.json", "r", encoding="utf-8") as fp:
    base_eval = json.load(fp)
print(f"Baseline Eval   : Accuracy = {base_eval['accuracy']:.4f}, Macro-F1 = {base_eval['macro_f1']:.4f}")

# 3. Huấn luyện và đánh giá Final Model trên tập eval
res_final = run_experiment(cfg_final, data)
save_result(res_final, f"{OUT_DIR}/results")
plot_run(res_final, f"{OUT_DIR}/figures/final-model.png")
final_eval(res_final["cfg"], res_final, data, f"{OUT_DIR}/predictions_eval.csv")
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_eval.csv", "--out", f"{OUT_DIR}/eval_result.json"], check=True)
with open(f"{OUT_DIR}/eval_result.json", "r", encoding="utf-8") as fp:
    final_eval = json.load(fp)
print(f"Final Model Eval: Accuracy = {final_eval['accuracy']:.4f}, Macro-F1 = {final_eval['macro_f1']:.4f}")
diff_f1 = final_eval['macro_f1'] - base_eval['macro_f1']
print(f"Cải thiện Macro-F1 so với baseline: +{diff_f1:.4f} (Vượt 2σ = {2*std_f1:.4f}: {diff_f1 > 2*std_f1})")


Cấu hình cuối cùng được chọn bằng Val: final-model (hparam-deep, val_macro_f1=0.8654)
Ghi du doan vao: ../predictions_baseline.csv (116203 dong)
Baseline Eval   : Accuracy = 0.9012, Macro-F1 = 0.8382
Ghi du doan vao: ../predictions_eval.csv (116203 dong)
Final Model Eval: Accuracy = 0.9127, Macro-F1 = 0.8643
Cải thiện Macro-F1 so với baseline: +0.0261 (Vượt 2σ = 0.0176: True)


In [7]:
# Phân tích lỗi theo lớp từ eval_result.json
with open(f"{OUT_DIR}/eval_result.json", "r", encoding="utf-8") as fp:
    eval_info = json.load(fp)

print("===== PHÂN TÍCH LỖI THEO LỚP TRÊN TẬP EVAL =====")
print(f"{'Lớp':<4} {'Support':<8} {'Precision':<10} {'Recall':<9} {'F1-Score':<8}")
for p in eval_info["per_class"]:
    print(f"{p['cls']:>3} {p['support']:>8} {p['precision']:>10.4f} {p['recall']:>9.4f} {p['f1']:>9.4f}")

print("\nMa trận nhầm lẫn:")
print(np.array(eval_info["confusion_matrix"]))


===== PHÂN TÍCH LỖI THEO LỚP TRÊN TẬP EVAL =====
Lớp  Support  Precision  Recall    F1-Score
  0    42368     0.9269  0.8922    0.9092
  1    56661     0.9167  0.9376    0.9270
  2     7151     0.8845  0.9232    0.9035
  3      549     0.8263  0.7541    0.7886
  4     1899     0.7266  0.8410    0.7796
  5     3473     0.8402  0.7889    0.8138
  6     4102     0.9349  0.9215    0.9282

Ma trận nhầm lẫn:
[[37801  4251     0     0    59    14   243]
 [ 2666 53123   161     5   508   178    20]
 [    3   153  6602    68    33   292     0]
 [    0     0   112   414     0    23     0]
 [   34   227    27     0  1597    14     0]
 [   17   139   562    14     1  2740     0]
 [  262    60     0     0     0     0  3780]]


**Phân tích lỗi (Error Analysis):**
- **Lớp khó nhất:** Lớp 4 (Aspen) có F1 thấp nhất ($0.7796$), kế tiếp là lớp 3 (Cottonwood/Willow) chỉ có 549 mẫu ($0.47\%$) với F1 = $0.7886$.
- **Nhầm lẫn chính:**
  - Lớp 0 và Lớp 1 (chiếm $85.2\%$ dữ liệu): 4 251 mẫu lớp 0 bị nhầm sang lớp 1 và 2 666 mẫu lớp 1 bị nhầm sang lớp 0 do sống ở các đai cao và thổ nhưỡng tương tự nhau tại vùng Rocky Mountains.
  - Lớp 4 (Aspen) bị nhầm 227 mẫu sang lớp 1 (Lodgepole Pine) do đặc tính phân bố xen kẽ trong tự nhiên.

In [8]:
# Xuất bảng experiments.xlsx
all_saved_results = load_results(f"{OUT_DIR}/results")
rows = []
for r in all_saved_results:
    eid = r["cfg"]["exp_id"]
    eval_score = None
    if eid in ["base-s1", "base-s2", "base-s3"]:
        eval_score = base_eval
    elif eid == "final-model":
        eval_score = final_eval
    n = "Baseline chuẩn seed 1" if eid == "base-s1" else ("Cấu hình cuối cùng tối ưu (chọn bằng val)" if eid == "final-model" else "")
    rows.append(to_row(r, eval_score, notes=n))

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
print(f"Đã xuất thành công bảng {OUT_DIR}/experiments.xlsx với {len(rows)} dòng thí nghiệm!")


Đã xuất thành công bảng ../experiments.xlsx với 24 dòng thí nghiệm!
